In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, detrend


def _safe_bandpass(x, fs, low=0.5, high=40.0, order=3):
    if len(x) < max(16, order * 3):
        return x.copy()

    nyq = 0.5 * fs
    low_n = max(low / nyq, 1e-6)
    high_n = min(high / nyq, 0.999)

    if low_n >= high_n:
        return x.copy()

    b, a = butter(order, [low_n, high_n], btype="band")
    return filtfilt(b, a, x)


def _safe_lowpass(x, fs, cutoff=15.0, order=3):
    if len(x) < max(16, order * 3):
        return x.copy()

    nyq = 0.5 * fs
    wn = min(cutoff / nyq, 0.999)
    if wn <= 0:
        return x.copy()

    b, a = butter(order, wn, btype="low")
    return filtfilt(b, a, x)


def _estimate_fs_from_time(time_s: np.ndarray) -> float:
    dt = np.diff(time_s)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) == 0:
        raise ValueError("無法從 time 估計取樣率")
    return 1.0 / np.median(dt)


def _window_quality_score(time_s: np.ndarray, x: np.ndarray) -> float:
    """
    分數越高代表該段越乾淨
    """
    if len(time_s) < 10 or len(x) < 10:
        return -np.inf

    # 檢查時間是否有效
    dt = np.diff(time_s)
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if len(dt) < 5:
        return -np.inf

    fs = 1.0 / np.median(dt)

    # 去趨勢 + bandpass
    x0 = detrend(x)
    x_bp = _safe_bandpass(x0, fs, low=0.5, high=40.0, order=3)

    # 低頻主體（保留 ECG 主體）
    x_main = _safe_lowpass(x_bp, fs, cutoff=15.0, order=3)

    # 高頻殘差（視為較偏噪聲）
    x_noise = x_bp - x_main

    # 穩定取樣分數
    dt_cv = np.std(dt) / (np.mean(dt) + 1e-12)
    score_dt = -5.0 * dt_cv

    # 離群值比例
    med = np.median(x_bp)
    mad = np.median(np.abs(x_bp - med)) + 1e-12
    robust_z = np.abs(x_bp - med) / (1.4826 * mad + 1e-12)
    outlier_ratio = np.mean(robust_z > 5.0)
    score_outlier = -3.0 * outlier_ratio

    # 主體訊號能量越大越好
    main_std = np.std(x_main)
    score_main = np.log(main_std + 1e-12)

    # 噪聲比例越小越好
    noise_std = np.std(x_noise)
    noise_ratio = noise_std / (main_std + 1e-12)
    score_noise = -np.log(noise_ratio + 1e-12)

    # 頻譜集中度（5~20 Hz 的 QRS 區域相對有能量較好）
    freqs = np.fft.rfftfreq(len(x_bp), d=1.0/fs)
    psd = np.abs(np.fft.rfft(x_bp)) ** 2

    total_band = (freqs >= 0.5) & (freqs <= 40.0)
    qrs_band = (freqs >= 5.0) & (freqs <= 20.0)

    total_power = np.sum(psd[total_band]) + 1e-12
    qrs_power = np.sum(psd[qrs_band])
    qrs_ratio = qrs_power / total_power
    score_spec = qrs_ratio

    score = (
        1.5 * score_main +
        2.0 * score_noise +
        1.5 * score_spec +
        score_dt +
        score_outlier
    )
    return float(score)


def extract_cleanest_30s(
    time_s: np.ndarray,
    lsb: np.ndarray,
    segment_sec: float = 30.0,
    step_sec: float = 1.0,
):
    """
    從整段 time/LSB 中找出最乾淨的 30 秒
    回傳:
        best_time_s, best_lsb, info
    """
    time_s = np.asarray(time_s, dtype=float)
    lsb = np.asarray(lsb, dtype=float)

    valid = np.isfinite(time_s) & np.isfinite(lsb)
    time_s = time_s[valid]
    lsb = lsb[valid]

    if len(time_s) < 10:
        raise ValueError("有效資料太少，無法擷取 30 秒片段")

    fs = _estimate_fs_from_time(time_s)
    win_n = int(round(segment_sec * fs))
    step_n = max(1, int(round(step_sec * fs)))

    if len(time_s) < win_n:
        raise ValueError(
            f"資料長度不足 {segment_sec} 秒，目前只有 {(time_s[-1] - time_s[0]):.2f} 秒"
        )

    best_score = -np.inf
    best_start = 0
    best_end = win_n

    for start in range(0, len(time_s) - win_n + 1, step_n):
        end = start + win_n
        t_seg = time_s[start:end]
        x_seg = lsb[start:end]

        # 避免視窗內時間跨度不足
        duration = t_seg[-1] - t_seg[0]
        if duration < segment_sec * 0.9:
            continue

        score = _window_quality_score(t_seg, x_seg)
        if score > best_score:
            best_score = score
            best_start = start
            best_end = end

    best_time = time_s[best_start:best_end].copy()
    best_lsb = lsb[best_start:best_end].copy()

    # 重新從 0 秒開始
    best_time = best_time - best_time[0]

    info = {
        "fs_est": fs,
        "segment_sec": segment_sec,
        "best_score": best_score,
        "start_idx": best_start,
        "end_idx": best_end,
        "orig_start_time": float(time_s[best_start]),
        "orig_end_time": float(time_s[best_end - 1]),
    }
    return best_time, best_lsb, info


def convert_ecg_csv_to_time_lsb(
    in_csv: str | Path,
    out_csv: str | Path,
    time_col: str = "time",
    ecg_col: str = "ecg",
    time_unit: str = "ns",   # 這批資料是 ns
    cleanest_30s: bool = False,
):
    in_csv = Path(in_csv)
    out_csv = Path(out_csv)

    df = pd.read_csv(in_csv)

    if time_col not in df.columns:
        raise ValueError(f"找不到時間欄位: {time_col}")
    if ecg_col not in df.columns:
        raise ValueError(f"找不到 ECG 欄位: {ecg_col}")

    # 只保留需要欄位，並轉成數值
    out_df = df[[time_col, ecg_col]].copy()
    out_df[time_col] = pd.to_numeric(out_df[time_col], errors="coerce")
    out_df[ecg_col] = pd.to_numeric(out_df[ecg_col], errors="coerce")

    # 去除無效值
    out_df = out_df.dropna(subset=[time_col, ecg_col]).reset_index(drop=True)

    if len(out_df) == 0:
        raise ValueError("資料清理後沒有可用資料")

    # 轉成相對秒數
    t0 = out_df[time_col].iloc[0]
    if time_unit == "ns":
        out_df["time"] = (out_df[time_col] - t0) / 1e9
    elif time_unit == "us":
        out_df["time"] = (out_df[time_col] - t0) / 1e6
    elif time_unit == "ms":
        out_df["time"] = (out_df[time_col] - t0) / 1e3
    elif time_unit == "s":
        out_df["time"] = out_df[time_col] - t0
    else:
        raise ValueError("time_unit 只能是 'ns', 'us', 'ms', 's'")

    out_df["LSB"] = out_df[ecg_col]

    # 只抓最乾淨的 30 秒
    if cleanest_30s:
        best_time, best_lsb, info = extract_cleanest_30s(
            out_df["time"].to_numpy(),
            out_df["LSB"].to_numpy(),
            segment_sec=30.08,
            step_sec=1.0,
        )
        out_df = pd.DataFrame({
            "time": best_time,
            "LSB": best_lsb
        })
        print(f"[{in_csv.name}] 最乾淨 30s 區段:")
        print(info)
    else:
        out_df = out_df[["time", "LSB"]]

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_csv, index=False, float_format="%.12f")

    print(f"已輸出: {out_csv}")
    print(out_df.head())


def batch_convert_ecg_folder(
    in_dir: str | Path,
    out_dir: str | Path,
    pattern: str = "sample_*.csv",
    cleanest_30s: bool = False,
):
    in_dir = Path(in_dir)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(in_dir.glob(pattern))
    if not files:
        print("找不到符合的檔案")
        return

    for f in files:
        if cleanest_30s:
            out_csv = out_dir / f"{f.stem}.csv"
        else:
            out_csv = out_dir / f.name

        convert_ecg_csv_to_time_lsb(
            f,
            out_csv,
            cleanest_30s=cleanest_30s
        )


if __name__ == "__main__":
    # 只輸出每個檔案最乾淨的 30 秒
    batch_convert_ecg_folder(
        in_dir="./ECG_polarH10",
        out_dir="./ECG",
        cleanest_30s=True,
    )

[sample_0.csv] 最乾淨 30s 區段:
{'fs_est': np.float64(130.09478836373344), 'segment_sec': 30.08, 'best_score': 0.27535294228954443, 'start_idx': 9880, 'end_idx': 13793, 'orig_start_time': 75.944269362, 'orig_end_time': 106.014683482}
已輸出: ECG\sample_0.csv
       time    LSB
0  0.000000 -0.339
1  0.007687 -0.332
2  0.015374 -0.332
3  0.023060 -0.329
4  0.030747 -0.329
[sample_1.csv] 最乾淨 30s 區段:
{'fs_est': np.float64(130.09125120726858), 'segment_sec': 30.08, 'best_score': -0.02467968824045, 'start_idx': 2340, 'end_idx': 6253, 'orig_start_time': 17.987436568, 'orig_end_time': 48.058665268}
已輸出: ECG\sample_1.csv
       time    LSB
0  0.000000  1.582
1  0.007687  1.553
2  0.015374  0.492
3  0.023060 -0.612
4  0.030747 -0.590
[sample_10.csv] 最乾淨 30s 區段:
{'fs_est': np.float64(130.07710450442954), 'segment_sec': 30.08, 'best_score': -0.20603619516889776, 'start_idx': 6370, 'end_idx': 10283, 'orig_start_time': 48.971131415, 'orig_end_time': 79.045541643}
已輸出: ECG\sample_10.csv
       time    LSB
0 